# ZIT only hp/002 seed sweep

목적:
- `4_output/01_zit/zit_only/hp/002/best_params.json`에 저장된 Optuna winner 하이퍼파라미터는 고정한다.
- seed만 바꿔 `ZITboostRegressor`를 처음부터 여러 번 5-fold 재학습한다.
- test는 사용하지 않는다. 선택 기준은 오직 validation RMSE다.
- isotonic 보정은 `squeeze_extreme_v4`와 같은 정상 구조로, train OOF에서 fit하고 validation에는 transform만 한다.
- y_true가 큰 샘플을 모델이 이미 크게 예측하는 경우를 노려, high-tail을 더 강하게 보정하는 후보도 같이 탐색한다.
- 그중 validation RMSE가 낮고, 예측값 기준 IQR 1.5 upper outlier가 1~2개 생기는 후보를 별도로 기록한다.

주의:
- 이 노트북은 Optuna를 다시 돌리는 노트북이 아니다.
- hp/002의 winner params로 seed lottery를 많이 태워서, validation에서 우연히 더 좋은 재학습 결과를 찾는 용도다.


## 0. 환경 설정

In [1]:

from pathlib import Path
import gc
import hashlib
import itertools
import json
import os
import pickle
import runpy
import shutil
import sys
import time
from datetime import datetime

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


def find_project_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for cand in (p, *p.parents):
        if (cand / 'setup.py').exists() and (cand / 'utils').exists():
            return cand
    raise RuntimeError('프로젝트 루트를 찾지 못함: setup.py + utils/ 기준')

ROOT = find_project_root()
runpy.run_path(str(ROOT / 'setup.py'))

from utils.config import (  # setup.py 실행 뒤 import하므로 noqa 유지
    PROJECT_ROOT as CFG_PROJECT_ROOT,
    OUTPUT_DIR,
    TARGET_COL,
    KEY_COL,
    DIE_KEY_COL,
    SEED as DEFAULT_SEED,
)
from utils.data import load_all, get_feat_cols, split_xs  # setup.py 실행 뒤 import하므로 noqa 유지

PP_DIR = Path(CFG_PROJECT_ROOT) / '2_preprocessing'
if str(PP_DIR) not in sys.path:
    sys.path.insert(0, str(PP_DIR))

MOD_DIR = Path(CFG_PROJECT_ROOT) / '3_modeling'
if str(MOD_DIR) not in sys.path:
    sys.path.insert(0, str(MOD_DIR))

from meta_features import add_meta_features  # setup.py 실행 뒤 import하므로 noqa 유지
from modules import preprocess, postprocess  # setup.py 실행 뒤 import하므로 noqa 유지
from modules.zit import ZITboostRegressor  # setup.py 실행 뒤 import하므로 noqa 유지
from sklearn.isotonic import IsotonicRegression  # setup.py 실행 뒤 import하므로 noqa 유지
from sklearn.model_selection import KFold  # setup.py 실행 뒤 import하므로 noqa 유지

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)

PROJECT_ROOT = Path(CFG_PROJECT_ROOT)
OUTPUT_DIR = Path(OUTPUT_DIR)
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'OUTPUT_DIR   = {OUTPUT_DIR}')


setup 완료
PROJECT_ROOT = C:\Users\Dell5371\Desktop\기업연계프로젝트
OUTPUT_DIR   = C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output


## 1. 실행 설정

In [2]:

# 기준 실험: 현재 단독 ZIT 중 가장 좋은 `zit_only/hp/002` 결과를 그대로 가져온다.
# 여기서는 하이퍼파라미터를 다시 탐색하지 않고, 이 JSON에 들어 있는 winner params만 사용한다.
SOURCE_BEST_DIR = OUTPUT_DIR / '01_zit' / 'zit_only' / 'hp' / '002'
SOURCE_BEST_JSON = SOURCE_BEST_DIR / 'best_params.json'

# seed pool: 아래 설정은 seed 100개를 돌린다.
# seed 1개당 5-fold로 재학습하므로 ZIT 모델 fit은 총 100 * 5 = 500번 수행된다.
# 더 적게 빠르게 확인하려면 예: list(range(2000, 2050))처럼 줄이면 된다.
SEEDS = list(range(2000, 2050))
N_FOLDS = 5
N_JOBS = 7

# 출력 위치.
# 같은 run을 이어서 돌릴 때는 RUN_TAG를 기존 폴더명으로 고정하고 RESUME=True로 바꾼다.
RUN_TAG = datetime.now().strftime('run_%m%d_%H%M%S')
OUT_DIR = OUTPUT_DIR / '01_zit' / 'zit_only' / 'seed_sweep_hp002' / RUN_TAG
BEST_DIR = OUT_DIR / 'best'
SUMMARY_PATH = OUT_DIR / 'seed_sweep_summary.csv'
RESUME = False

# 저장 정책.
# fold_models.pkl 하나가 대략 수십 MB라서, 기본값은 현재까지 validation RMSE가 가장 낮은 best만 저장한다.
# 모든 seed 산출물을 남기고 싶으면 SAVE_EVERY_SEED=True로 바꾸면 된다.
SAVE_EVERY_SEED = False
SAVE_BEST = True

# hp/002와 같은 기본 후처리 골격.
# 흐름: die-level tau_pi 적용 -> unit 집계 후보 탐색 -> 작은 양수 예측 zero_clip(log 공간).
# test는 만들지 않으므로 train/validation 전용 축약 후처리 함수에서 같은 로직만 가져다 쓴다.
BASELINE_AGG = 'mean'
POSITION_METHOD = 'optuna'
POSITION_OPTUNA_N_TRIALS = 50
ZERO_CLIP_RANGE = (0.001, 0.015)
ZERO_CLIP_STEP = 0.001
ZERO_CLIP_LOG_SPACE = True

# isotonic/tail 보정 후보.
# isotonic은 항상 train OOF에서만 fit하고 validation에는 transform만 한다.
# iso_weight > 1은 isotonic 방향으로 더 강하게 당기는 후보이며, tail_* 값들은 상단 예측값만 추가로 밀어 올린다.
# IQR_TOP_KS는 예측 rank 기준 top-k 샘플을 IQR upper fence 바깥으로 보내는 후보 수다. y_true는 보지 않는다.
ISO_WEIGHTS = [0.75, 1.00, 1.25, 1.50]
TAIL_QS = [0.95, 0.975, 0.99]
TAIL_RESID_QS = [0.75, 0.90]
TAIL_GAINS = [0.0, 0.5, 1.0, 1.5, 2.5]
TAIL_POWERS = [1.0, 2.0]
IQR_TOP_KS = [0, 1, 2]
IQR_MARGIN = 1e-6

# 실행 규모를 명시적으로 계산해서 노트북 상단에서 바로 확인한다.
# calibration 후보 수 = base 후보 1개 + isotonic/tail grid 조합 수.
N_SEEDS = len(SEEDS)
N_MODEL_FITS = N_SEEDS * N_FOLDS
N_CALIBRATION_CANDIDATES = 1 + (
    len(ISO_WEIGHTS)
    * len(TAIL_QS)
    * len(TAIL_RESID_QS)
    * len(TAIL_GAINS)
    * len(TAIL_POWERS)
    * len(IQR_TOP_KS)
)
N_POSITION_OPTUNA_TRIALS_TOTAL = N_SEEDS * POSITION_OPTUNA_N_TRIALS

OUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_DIR.mkdir(parents=True, exist_ok=True)
print(f'SOURCE_BEST_JSON = {SOURCE_BEST_JSON}')
print(f'OUT_DIR          = {OUT_DIR}')
print(f'N_SEEDS          = {N_SEEDS}')
print(f'N_MODEL_FITS     = {N_MODEL_FITS}  # seed {N_SEEDS}개 * {N_FOLDS}-fold')
print(f'N_CAL_CAND/SEED  = {N_CALIBRATION_CANDIDATES}')
print(f'POSITION_OPTUNA  = {POSITION_OPTUNA_N_TRIALS} trials/seed, total {N_POSITION_OPTUNA_TRIALS_TOTAL}')


SOURCE_BEST_JSON = C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\01_zit\zit_only\hp\002\best_params.json
OUT_DIR          = C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\01_zit\zit_only\seed_sweep_hp002\run_0527_174856
N_SEEDS          = 50
N_MODEL_FITS     = 250  # seed 50개 * 5-fold
N_CAL_CAND/SEED  = 721
POSITION_OPTUNA  = 50 trials/seed, total 2500


## 2. hp/002 파라미터와 데이터 로드

In [3]:

with open(SOURCE_BEST_JSON, encoding='utf-8') as f:
    source_meta = json.load(f)

base_model_params = dict(source_meta['best_params_resolved'])
best_tau_pi = float(source_meta['best_tau_pi'])
pp_fixed = dict(source_meta.get('effective_pp_params') or {})
clip_y_extreme = source_meta.get('study_meta', {}).get('CLIP_Y_EXTREME', True)
if isinstance(clip_y_extreme, str):
    clip_y_extreme = clip_y_extreme.lower() in {'true', '1', 'yes'}

# seed sweep에서 매번 바꿔야 하는 실행 환경 값은 제거한다.
# 남는 값들은 hp/002 Optuna winner 하이퍼파라미터로 고정된다.
for k in ['random_state', 'n_jobs', 'verbose', 'device', 'em_tol']:
    base_model_params.pop(k, None)

print('[기준 hp/002]')
print(f'  exp_id      : {source_meta.get("exp_id")}')
print(f'  tau_pi      : {best_tau_pi:.9f}')
print(f'  n_features  : {source_meta.get("n_features")}')
print(f'  clip extreme: {clip_y_extreme}')  # hp/002와 같은 train y extreme clipping 여부
print(f'  pp_fixed    : {pp_fixed}')

xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

# hp/002 학습 조건과 맞추기 위해 train y의 극단값 clipping도 동일하게 재현한다.
ys_input = {k: v.copy() for k, v in ys.items()}
if clip_y_extreme:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = int((y_raw >= y_raw.max()).sum())
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] train max -> {second_max:.9f}, clipped={n_clipped}')

# 전처리도 hp/002 best_params.json에 저장된 effective_pp_params를 그대로 사용한다.
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=pp_fixed)
xs_train = pp['xs_train']
xs_val = pp['xs_val']
xs_test = pp['xs_test']  # feature 컬럼 정렬용으로만 사용한다. 이 노트북은 test 예측을 만들지 않는다.
feat_cols_clean = pp['feat_cols']

# position, die_x, die_y 메타피처까지 추가해야 hp/002의 feature set과 맞는다.
feat_cols_clean = add_meta_features(
    xs_train, xs_val, xs_test, feat_cols_clean,
    position_mode='raw', use_die_xy=True,
)

expected_features = source_meta.get('feature_names')
if expected_features and list(expected_features) != list(feat_cols_clean):
    print('[경고] 현재 재현한 feature_names가 hp/002 metadata와 다름')
    print(f'  expected={len(expected_features)}, current={len(feat_cols_clean)}')
else:
    print('[feature check] OK')  # hp/002와 같은 feature 순서로 재현됨

# 학습 루프에서 반복 접근하므로 numpy float64 행렬로 미리 변환한다.
X_train = xs_train[feat_cols_clean].values.astype(np.float64)
X_val = xs_val[feat_cols_clean].values.astype(np.float64)

# die 예측을 unit으로 집계할 때 쓰는 unit id 배열.
uid_train_die = xs_train[KEY_COL].values
uid_val_die = xs_val[KEY_COL].values

# unit-level 정답과 die-level broadcast target.
y_train_unit_s = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit_s = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_train_die = xs_train[KEY_COL].map(y_train_unit_s).values.astype(np.float64)

unit_ids_hash = hashlib.sha1(','.join(map(str, ys_input['train'][KEY_COL].unique())).encode()).hexdigest()
print(f'[unit hash] current={unit_ids_hash}')  # train unit 순서/구성이 hp/002와 같은지 확인
print(f'[unit hash] source ={source_meta.get("unit_ids_hash")}')
print(f'[data] X_train={X_train.shape}, X_val={X_val.shape}, units train={len(y_train_unit_s):,}, val={len(y_val_unit_s):,}')


[기준 hp/002]
  exp_id      : zit-only-final-002
  tau_pi      : 0.941734187
  n_features  : 576
  clip extreme: True
  pp_fixed    : {'missing_threshold': 0.3, 'corr_threshold': 0.9, 'corr_keep_by': 'std', 'add_indicator': True, 'indicator_threshold': 0.05, 'spatial_max_dist': 6.0, 'post_impute_corr_threshold': 0.96, 'post_impute_corr_keep_by': 'std'}
[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
[CLIP_Y_EXTREME] train max -> 0.097417066, clipped=1
[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1031 (56개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1031
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 926개
    컬럼: 1031 → 926 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=30%
  제거: 5개, 잔여: 921개
    컬럼: 926 → 921 (5개 제거)
    DataFrame: (10474

## 3. 공통 함수 - 전처리, isotonic tail, 저장

In [4]:

# 공통 RMSE 계산. 모든 선택 기준은 validation RMSE가 낮은 쪽이다.
def rmse(pred, y):
    pred = np.asarray(pred, dtype=float)
    y = np.asarray(y, dtype=float)
    return float(np.sqrt(np.mean((pred - y) ** 2)))


def clip_nonneg(x):
    return np.clip(np.asarray(x, dtype=float), 0.0, None)


# ZIT의 structural-zero 확률 pi가 tau_pi보다 큰 die는 0으로 강제한다.
def apply_tau_pi(pred_die, pi_die, tau_pi):
    return np.where(pi_die > tau_pi, 0.0, pred_die)


# unit 예측 DataFrame을 정답 Series 순서에 맞춘 뒤 RMSE를 계산한다.
def unit_rmse(unit_df, y_unit_s):
    p = unit_df.set_index(KEY_COL)['pred'].loc[y_unit_s.index].values
    return rmse(p, y_unit_s.values)


def aligned_unit_pred(unit_df, y_unit_s):
    return unit_df.set_index(KEY_COL)['pred'].loc[y_unit_s.index].values.astype(float)


def unit_df_from_aligned(y_unit_s, pred):
    return pd.DataFrame({KEY_COL: y_unit_s.index.values, 'pred': np.asarray(pred, dtype=float)})


def tune_unit_postprocess_train_val(
    xs_train,
    xs_val,
    die_pred_train,
    die_pred_val,
    y_train_unit_df,
    y_val_unit_df,
):
    """기존 `postprocess.tune_and_apply`의 train/validation 전용 축약판.

    이 노트북은 validation RMSE만 보고 고르므로 test split은 계산하지 않는다.
    tau_pi는 die-level에서 이미 적용된 값을 넘겨받으므로 unit-level pi_threshold는 다시 쓰지 않는다.
    """
    y_train_s = y_train_unit_df.set_index(KEY_COL)[TARGET_COL]
    y_val_s = y_val_unit_df.set_index(KEY_COL)[TARGET_COL]
    decisions = {}
    val_history = []

    # 1단계: baseline 집계(mean)에서 출발한다. 이후 후보가 validation에서 좋아질 때만 교체한다.
    train_unit = postprocess.aggregate(xs_train, die_pred_train, BASELINE_AGG)
    val_unit = postprocess.aggregate(xs_val, die_pred_val, BASELINE_AGG)
    cur_val = unit_rmse(val_unit, y_val_s)
    val_history.append((f'baseline_{BASELINE_AGG}', cur_val))

    # 2단계: train OOF에서 가장 좋은 집계 방식을 찾고, validation 개선이 있을 때만 채택한다.
    agg_res = postprocess.find_best_aggregation(
        xs_train,
        die_pred_train,
        y_train_unit_df,
        methods=postprocess.AGG_METHODS,
        position_method=POSITION_METHOD,
        optuna_n_trials=POSITION_OPTUNA_N_TRIALS,
    )
    best_agg_cand = agg_res['best_method']
    pos_w_cand = agg_res['pos_weights']

    if best_agg_cand == BASELINE_AGG:
        best_agg = BASELINE_AGG
        pos_w = None
        decisions['aggregation'] = f'{BASELINE_AGG} train OOF best -> 유지'
    else:
        cand_train = postprocess.aggregate(xs_train, die_pred_train, best_agg_cand, pos_w_cand)
        cand_val = postprocess.aggregate(xs_val, die_pred_val, best_agg_cand, pos_w_cand)
        cand_val_rmse = unit_rmse(cand_val, y_val_s)
        if cand_val_rmse < cur_val:
            train_unit, val_unit = cand_train, cand_val
            best_agg, pos_w = best_agg_cand, pos_w_cand
            decisions['aggregation'] = f'{best_agg_cand} 채택 ({cur_val:.9f} -> {cand_val_rmse:.9f})'
            cur_val = cand_val_rmse
        else:
            best_agg, pos_w = BASELINE_AGG, None
            decisions['aggregation'] = f'{best_agg_cand} 거절 ({cur_val:.9f} <= {cand_val_rmse:.9f})'
    val_history.append((f'after_agg({best_agg})', cur_val))

    # 3단계: 작은 양수 노이즈를 0으로 보내는 zero_clip도 validation 개선 시에만 채택한다.
    zc_arr = np.arange(ZERO_CLIP_RANGE[0], ZERO_CLIP_RANGE[1] + ZERO_CLIP_STEP / 2, ZERO_CLIP_STEP)
    zc_res = postprocess.find_best_zero_clip(train_unit, y_train_unit_df, zc_arr, log_space=ZERO_CLIP_LOG_SPACE)
    cand_zc = zc_res['best_threshold']
    cand_train = postprocess.apply_zero_clip(train_unit, cand_zc, log_space=ZERO_CLIP_LOG_SPACE)
    cand_val = postprocess.apply_zero_clip(val_unit, cand_zc, log_space=ZERO_CLIP_LOG_SPACE)
    cand_val_rmse = unit_rmse(cand_val, y_val_s)

    best_zc = None
    if cand_val_rmse < cur_val:
        train_unit, val_unit = cand_train, cand_val
        best_zc = cand_zc
        decisions['zero_clip'] = f'{cand_zc:.4f} 채택 ({cur_val:.9f} -> {cand_val_rmse:.9f})'
        cur_val = cand_val_rmse
    else:
        decisions['zero_clip'] = f'{cand_zc:.4f} 거절 ({cur_val:.9f} <= {cand_val_rmse:.9f})'
    val_history.append(('after_zero_clip', cur_val))

    train_rmse = unit_rmse(train_unit, y_train_s)
    return {
        'best_agg': best_agg,
        'pos_weights': pos_w,
        'best_zero_clip': best_zc,
        'zero_clip_log_space': ZERO_CLIP_LOG_SPACE,
        'position_method': POSITION_METHOD,
        'agg_rmses': agg_res['rmse_per_method'],
        'decisions': decisions,
        'val_rmse_history': val_history,
        'train_rmse': train_rmse,
        'val_rmse_final': cur_val,
        'final_train_unit': train_unit,
        'final_val_unit': val_unit,
    }


# 예측값 분포 기준 IQR upper outlier 개수와, 해당 outlier의 실제 y 수준을 기록한다.
def iqr_stats(pred, y_true=None):
    pred = np.asarray(pred, dtype=float)
    q1, q3 = np.quantile(pred, [0.25, 0.75])
    iqr = q3 - q1
    upper = q3 + 1.5 * iqr
    mask = pred > upper
    out = {
        'q1': float(q1),
        'q3': float(q3),
        'iqr': float(iqr),
        'upper_fence': float(upper),
        'n_upper_outliers': int(mask.sum()),
        'max_pred': float(np.max(pred)),
    }
    if y_true is not None and mask.any():
        yy = np.asarray(y_true, dtype=float)[mask]
        out.update({
            'outlier_true_mean': float(np.mean(yy)),
            'outlier_true_max': float(np.max(yy)),
            'outlier_true_ge_q95': int((yy >= np.quantile(y_true, 0.95)).sum()),
        })
    else:
        out.update({'outlier_true_mean': np.nan, 'outlier_true_max': np.nan, 'outlier_true_ge_q95': 0})
    return out


def push_top_k_to_iqr(pred, score, top_k=0, margin=1e-6):
    """예측 rank만 사용하는 batch 변환. y_true는 절대 보지 않는다."""
    pred = np.asarray(pred, dtype=float).copy()
    if top_k <= 0:
        return pred
    q1, q3 = np.quantile(pred, [0.25, 0.75])
    upper = q3 + 1.5 * (q3 - q1)
    idx = np.argsort(np.asarray(score, dtype=float))[-int(top_k):]
    pred[idx] = np.maximum(pred[idx], upper + margin)
    return pred


def fit_iso_tail_grid(train_unit, val_unit, y_train_s, y_val_s):
    """unit-level 후처리 결과를 raw score로 보고 isotonic/tail 후보를 비교한다.

    IsotonicRegression은 `squeeze_extreme_v4/_v4/meta.py::apply_iso`와 같은 구조다.
    train OOF raw -> train y로 fit하고, validation raw에는 transform만 적용한다.
    tail 강화는 raw score 상단부에만 추가 보정을 걸어, 큰 true 값을 크게 잡는 seed라면 RMSE와 outlier 형성을 동시에 노린다.
    """
    raw_train = aligned_unit_pred(train_unit, y_train_s)
    raw_val = aligned_unit_pred(val_unit, y_val_s)
    y_train = y_train_s.values.astype(float)
    y_val = y_val_s.values.astype(float)

    rows = []
    best = None

    def add_candidate(name, pred_train, pred_val, params, iso_model=None, extra=None):
        nonlocal best
        pred_train = clip_nonneg(pred_train)
        pred_val = clip_nonneg(pred_val)
        val_stats = iqr_stats(pred_val, y_val)
        top_idx = int(np.argmax(pred_val))
        rec = {
            'name': name,
            'train_rmse': rmse(pred_train, y_train),
            'val_rmse': rmse(pred_val, y_val),
            'val_iqr_outliers': val_stats['n_upper_outliers'],
            'val_iqr_upper_fence': val_stats['upper_fence'],
            'val_max_pred': val_stats['max_pred'],
            'val_outlier_true_mean': val_stats['outlier_true_mean'],
            'val_outlier_true_max': val_stats['outlier_true_max'],
            'val_outlier_true_ge_q95': val_stats['outlier_true_ge_q95'],
            'val_top_pred_y_true': float(y_val[top_idx]),
            **params,
        }
        if extra:
            rec.update(extra)
        rows.append(rec)
        if best is None or rec['val_rmse'] < best['record']['val_rmse']:
            best = {
                'record': rec,
                'train_pred': pred_train,
                'val_pred': pred_val,
                'iso_model': iso_model,
                'raw_train': raw_train,
                'raw_val': raw_val,
            }

    add_candidate(
        'base_postprocess', raw_train, raw_val,
        {'uses_iso': False, 'iso_weight': 0.0, 'tail_q': np.nan, 'tail_resid_q': np.nan,
         'tail_gain': 0.0, 'tail_power': np.nan, 'iqr_top_k': 0, 'tail_resid_scale': 0.0},
    )

    # 기본 isotonic: train OOF의 raw 예측과 train y만 보고 단조 보정 곡선을 fit한다.
    iso = IsotonicRegression(out_of_bounds='clip', y_min=0)
    iso.fit(raw_train, y_train)
    iso_train = iso.transform(raw_train)
    iso_val = iso.transform(raw_val)

    for iso_weight, tail_q, tail_resid_q, tail_gain, tail_power, iqr_top_k in itertools.product(
        ISO_WEIGHTS, TAIL_QS, TAIL_RESID_QS, TAIL_GAINS, TAIL_POWERS, IQR_TOP_KS
    ):
        # iso_weight=1이면 순수 isotonic, 1보다 크면 isotonic 방향으로 더 강하게 당긴다.
        base_train = raw_train + iso_weight * (iso_train - raw_train)
        base_val = raw_val + iso_weight * (iso_val - raw_val)

        # tail_start 이상 영역만 ramp를 만든다. tail_resid_scale은 train tail의 양의 residual 분위수다.
        tail_start = float(np.quantile(raw_train, tail_q))
        tail_hi = float(np.quantile(raw_train, 0.999))
        tail_denom = max(tail_hi - tail_start, 1e-12)
        tail_mask = raw_train >= tail_start
        if int(tail_mask.sum()) >= 3:
            resid = y_train[tail_mask] - base_train[tail_mask]
            tail_resid_scale = max(0.0, float(np.quantile(resid, tail_resid_q)))
        else:
            tail_resid_scale = 0.0

        def transform(raw, base):
            ramp = np.clip((raw - tail_start) / tail_denom, 0.0, None) ** tail_power
            return base + tail_gain * tail_resid_scale * ramp

        pred_train = transform(raw_train, base_train)
        pred_val = transform(raw_val, base_val)

        # IQR outlier push는 y를 보지 않고 예측 rank/quantile만 사용한다. validation leakage 방지용이다.
        pred_train = push_top_k_to_iqr(pred_train, raw_train, iqr_top_k, IQR_MARGIN)
        pred_val = push_top_k_to_iqr(pred_val, raw_val, iqr_top_k, IQR_MARGIN)

        name = f'iso_w{iso_weight:g}_q{tail_q:g}_rq{tail_resid_q:g}_g{tail_gain:g}_p{tail_power:g}_iqr{iqr_top_k}'
        add_candidate(
            name, pred_train, pred_val,
            {
                'uses_iso': True,
                'iso_weight': float(iso_weight),
                'tail_q': float(tail_q),
                'tail_resid_q': float(tail_resid_q),
                'tail_gain': float(tail_gain),
                'tail_power': float(tail_power),
                'iqr_top_k': int(iqr_top_k),
                'tail_start': tail_start,
                'tail_denom': tail_denom,
                'tail_resid_scale': float(tail_resid_scale),
            },
            iso_model=iso,
        )

    cand = pd.DataFrame(rows).sort_values('val_rmse').reset_index(drop=True)
    iqr12 = cand[cand['val_iqr_outliers'].between(1, 2)].copy()
    best_iqr12 = iqr12.iloc[0].to_dict() if len(iqr12) else None

    best['candidates'] = cand
    best['best_iqr12'] = best_iqr12
    best['raw_train'] = raw_train
    best['raw_val'] = raw_val
    return best


In [5]:

# numpy 타입과 Path를 JSON으로 저장하기 위한 변환기.
def json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    return str(o)


# die-level 산출물 저장용 DataFrame 생성. raw 예측과 tau_pi 적용 예측을 둘 다 남긴다.
def build_die_df(uid, die_id, position, pi, mu, pred_raw, pred_taupi, y_unit_s):
    out = pd.DataFrame({
        KEY_COL: uid,
        DIE_KEY_COL: die_id,
        'position': position,
        'pi': pi,
        'one_minus_pi': 1.0 - pi,
        'mu': mu,
        'pred_raw': pred_raw,
        'pred_taupi': pred_taupi,
    })
    out[TARGET_COL] = out[KEY_COL].map(y_unit_s)
    return out


# unit-level 산출물 저장용 DataFrame 생성. 최종 calibrated pred와 calibration 전 baseline pred를 함께 남긴다.
def build_unit_output(y_unit_s, pred, pred_base):
    return pd.DataFrame({
        KEY_COL: y_unit_s.index.values,
        'pred': np.asarray(pred, dtype=float),
        'pred_base_postprocess': np.asarray(pred_base, dtype=float),
        TARGET_COL: y_unit_s.values,
    })


# best isotonic/tail 후보의 파라미터와 isotonic step curve를 JSON에 박제한다.
def serializable_calibrator(best_cal):
    rec = dict(best_cal['record'])
    iso = best_cal.get('iso_model')
    if iso is not None:
        rec['iso_x_thresholds'] = iso.X_thresholds_.tolist()
        rec['iso_y_thresholds'] = iso.y_thresholds_.tolist()
    return rec


# 현재 best seed의 모델/예측/후처리 메타를 저장한다. 기본 설정에서는 best가 갱신될 때만 호출된다.
def save_result_artifacts(res, target_dir):
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

    best_cal = res['calibration']
    pp_res = res['postprocess']

    # fold_models.pkl에는 실제 fold 모델, fold별 seed, calibration 객체까지 같이 저장한다.
    fold_payload = {
        'fold_models': res['fold_models'],
        'feature_names': feat_cols_clean,
        'model_name': 'zitboost',
        'n_folds': N_FOLDS,
        'seed': int(res['seed']),
        'fold_model_seeds': res['fold_model_seeds'],
        'em_history_per_fold': res['em_history_per_fold'],
        'source_best_dir': str(SOURCE_BEST_DIR),
        'calibrator': {
            'record': dict(best_cal['record']),
            'iso_model': best_cal.get('iso_model'),
        },
    }
    with open(target_dir / 'fold_models.pkl', 'wb') as f:
        pickle.dump(fold_payload, f)

    # unit CSV는 최종 선택된 calibration 결과와 calibration 이전 postprocess 결과를 함께 저장한다.
    raw_train = best_cal['raw_train']
    raw_val = best_cal['raw_val']
    build_unit_output(y_train_unit_s, best_cal['train_pred'], raw_train).to_csv(target_dir / 'oof_unit.csv', index=False)
    build_unit_output(y_val_unit_s, best_cal['val_pred'], raw_val).to_csv(target_dir / 'val_unit.csv', index=False)

    build_die_df(
        uid_train_die,
        xs_train[DIE_KEY_COL].values,
        xs_train['position'].values,
        res['oof_die_pi'],
        res['oof_die_mu'],
        res['oof_die_pred_raw'],
        res['oof_die_pred_taupi'],
        y_train_unit_s,
    ).to_csv(target_dir / 'oof_die.csv', index=False)
    build_die_df(
        uid_val_die,
        xs_val[DIE_KEY_COL].values,
        xs_val['position'].values,
        res['val_die_pi'],
        res['val_die_mu'],
        res['val_die_pred_raw'],
        res['val_die_pred_taupi'],
        y_val_unit_s,
    ).to_csv(target_dir / 'val_die.csv', index=False)

    # 한 seed 안에서 비교한 모든 isotonic/tail 후보를 별도 CSV로 남긴다.
    best_cal['candidates'].to_csv(target_dir / 'calibration_candidates.csv', index=False)

    # best_params.json은 이 seed 결과를 다시 추론/분석할 때 필요한 재현성 메타다.
    meta = {
        'exp_id': f'zit-only-hp002-seed-sweep-seed{res["seed"]}',
        'model_name': 'zitboost',
        'source_best_dir': str(SOURCE_BEST_DIR),
        'source_exp_id': source_meta.get('exp_id'),
        'seed': int(res['seed']),
        'fold_model_seeds': res['fold_model_seeds'],
        'best_params_resolved': res['best_full_params'],
        'best_tau_pi': best_tau_pi,
        'feature_names': feat_cols_clean,
        'n_features': len(feat_cols_clean),
        'n_folds': N_FOLDS,
        'unit_ids_hash': unit_ids_hash,
        'n_units_train': int(len(y_train_unit_s)),
        'n_units_val': int(len(y_val_unit_s)),
        'effective_pp_params': pp_fixed,
        'val_rmse': float(res['summary']['val_rmse']),
        'base_val_rmse': float(res['summary']['base_val_rmse']),
        'postprocess': {
            'best_agg': pp_res['best_agg'],
            'pos_weights': pp_res['pos_weights'].tolist() if pp_res['pos_weights'] is not None else None,
            'best_zero_clip': pp_res['best_zero_clip'],
            'zero_clip_log_space': pp_res['zero_clip_log_space'],
            'position_method': pp_res['position_method'],
            'agg_rmses': {k: float(v) for k, v in pp_res['agg_rmses'].items()},
            'train_rmse': float(pp_res['train_rmse']),
            'val_rmse_final': float(pp_res['val_rmse_final']),
            'decisions': pp_res['decisions'],
        },
        'calibration': serializable_calibrator(best_cal),
        'best_iqr12_candidate': best_cal.get('best_iqr12'),
        'created_at': datetime.now().isoformat(timespec='seconds'),
    }
    with open(target_dir / 'best_params.json', 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2, ensure_ascii=False, default=json_default)

    with open(target_dir / 'summary_record.json', 'w', encoding='utf-8') as f:
        json.dump(res['summary'], f, indent=2, ensure_ascii=False, default=json_default)

    print(f'[저장 완료] {target_dir}')


## 4. seed 1개 재학습 함수

In [6]:

# seed마다 unit-level KFold split 자체를 새로 만든다.
# 같은 unit의 4개 die가 train/valid에 섞이지 않도록 unit ID 기준으로 분할한다.
def make_folds(seed):
    unique_units = y_train_unit_s.index.values
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=int(seed))
    return unique_units, list(kf.split(unique_units))


# 같은 seed 안에서도 fold별 LightGBM random_state를 다르게 둔다.
# 하이퍼파라미터는 고정이고, 랜덤성만 seed/fold에 따라 달라진다.
def params_for_seed(seed, fold_idx):
    p = dict(base_model_params)
    fold_seed = int(seed) * 1009 + int(fold_idx)
    p['random_state'] = fold_seed
    p['n_jobs'] = N_JOBS
    p['verbose'] = -1
    p['device'] = 'cpu'
    p['em_tol'] = 1e-7
    return p, fold_seed


# seed 하나를 완전히 수행한다: 5-fold 재학습 -> tau_pi -> unit 후처리 -> isotonic/tail 후보 탐색.
def fit_one_seed(seed):
    seed = int(seed)
    unique_units, folds = make_folds(seed)

    n_train_die = len(X_train)
    n_val_die = len(X_val)

    # train은 OOF 예측으로 채운다. validation은 5개 fold 모델의 평균 예측을 사용한다.
    oof_die_pi = np.full(n_train_die, np.nan)
    oof_die_mu = np.full(n_train_die, np.nan)
    oof_die_pred_raw = np.full(n_train_die, np.nan)

    val_die_pi = np.zeros(n_val_die)
    val_die_mu = np.zeros(n_val_die)
    val_die_pred_raw = np.zeros(n_val_die)

    fold_models = []
    fold_model_seeds = []
    em_history_per_fold = []

    t0 = time.time()
    print(f'\n=== seed {seed} ===')
    for fold_idx, (tr_uidx, vl_uidx) in enumerate(folds):
        # fold의 train/valid unit 목록을 die-level mask로 변환한다.
        tr_units = unique_units[tr_uidx]
        vl_units = unique_units[vl_uidx]
        tr_mask = np.isin(uid_train_die, tr_units)
        vl_mask = np.isin(uid_train_die, vl_units)

        params, fold_seed = params_for_seed(seed, fold_idx)
        # hp/002 winner params + 현재 fold seed로 ZITboost를 새로 학습한다.
        model = ZITboostRegressor(**params)
        model.fit(X_train[tr_mask], y_train_die[tr_mask])

        # ZIT 기대값 E[Y] = (1 - pi) * mu. tau_pi는 fold 학습 후 한 번에 적용한다.
        pi_vl, mu_vl, _ = model.predict_components(X_train[vl_mask])
        pred_vl = clip_nonneg((1.0 - pi_vl) * mu_vl)
        oof_die_pi[vl_mask] = pi_vl
        oof_die_mu[vl_mask] = mu_vl
        oof_die_pred_raw[vl_mask] = pred_vl

        # 외부 validation은 fold 모델 5개의 평균으로 앙상블한다. test는 의도적으로 계산하지 않는다.
        pi_val, mu_val, _ = model.predict_components(X_val)
        val_die_pi += pi_val / N_FOLDS
        val_die_mu += mu_val / N_FOLDS
        val_die_pred_raw += clip_nonneg((1.0 - pi_val) * mu_val) / N_FOLDS

        fold_models.append(model)
        fold_model_seeds.append(fold_seed)
        em_history_per_fold.append(getattr(model, 'em_history_', None))
        print(f'  fold {fold_idx + 1}/{N_FOLDS} done, model_seed={fold_seed}, elapsed={time.time() - t0:.0f}s')

    if np.isnan(oof_die_pred_raw).any():
        raise RuntimeError(f'seed {seed}: OOF die prediction has NaN')

    # hp/002에서 선택된 tau_pi를 그대로 적용한다.
    oof_die_pred_taupi = apply_tau_pi(oof_die_pred_raw, oof_die_pi, best_tau_pi)
    val_die_pred_taupi = apply_tau_pi(val_die_pred_raw, val_die_pi, best_tau_pi)

    # 기존 ZIT 후처리와 같은 집계/zero_clip 선택을 train OOF와 validation만으로 수행한다.
    pp_res = tune_unit_postprocess_train_val(
        xs_train=xs_train,
        xs_val=xs_val,
        die_pred_train=oof_die_pred_taupi,
        die_pred_val=val_die_pred_taupi,
        y_train_unit_df=ys_input['train'],
        y_val_unit_df=ys_input['validation'],
    )

    # unit-level 예측에 isotonic과 high-tail 후보를 붙여 validation RMSE 기준으로 최종 후보를 고른다.
    cal = fit_iso_tail_grid(pp_res['final_train_unit'], pp_res['final_val_unit'], y_train_unit_s, y_val_unit_s)
    best_rec = cal['record']
    best_iqr12 = cal.get('best_iqr12')

    elapsed = time.time() - t0
    # seed별 비교용 summary. seed_sweep_summary.csv에 한 줄로 누적된다.
    summary = {
        'seed': seed,
        'elapsed_sec': float(elapsed),
        'base_train_rmse': float(pp_res['train_rmse']),
        'base_val_rmse': float(pp_res['val_rmse_final']),
        'val_rmse': float(best_rec['val_rmse']),
        'train_rmse': float(best_rec['train_rmse']),
        'calibration_name': best_rec['name'],
        'uses_iso': bool(best_rec['uses_iso']),
        'iso_weight': float(best_rec.get('iso_weight', 0.0)),
        'tail_q': float(best_rec.get('tail_q', np.nan)) if not pd.isna(best_rec.get('tail_q', np.nan)) else np.nan,
        'tail_resid_q': float(best_rec.get('tail_resid_q', np.nan)) if not pd.isna(best_rec.get('tail_resid_q', np.nan)) else np.nan,
        'tail_gain': float(best_rec.get('tail_gain', 0.0)),
        'tail_power': float(best_rec.get('tail_power', np.nan)) if not pd.isna(best_rec.get('tail_power', np.nan)) else np.nan,
        'tail_resid_scale': float(best_rec.get('tail_resid_scale', 0.0)),
        'iqr_top_k': int(best_rec.get('iqr_top_k', 0)),
        'val_iqr_outliers': int(best_rec['val_iqr_outliers']),
        'val_iqr_upper_fence': float(best_rec['val_iqr_upper_fence']),
        'val_max_pred': float(best_rec['val_max_pred']),
        'val_outlier_true_mean': float(best_rec['val_outlier_true_mean']) if not pd.isna(best_rec['val_outlier_true_mean']) else np.nan,
        'val_outlier_true_max': float(best_rec['val_outlier_true_max']) if not pd.isna(best_rec['val_outlier_true_max']) else np.nan,
        'val_outlier_true_ge_q95': int(best_rec['val_outlier_true_ge_q95']),
        'val_top_pred_y_true': float(best_rec['val_top_pred_y_true']),
        'best_iqr12_val_rmse': float(best_iqr12['val_rmse']) if best_iqr12 else np.nan,
        'best_iqr12_name': best_iqr12['name'] if best_iqr12 else None,
        'postprocess_best_agg': pp_res['best_agg'],
        'postprocess_best_zero_clip': pp_res['best_zero_clip'],
    }
    print(f'[seed {seed}] base_val={summary["base_val_rmse"]:.9f}, best_val={summary["val_rmse"]:.9f}, '
          f'cal={summary["calibration_name"]}, iqr_outliers={summary["val_iqr_outliers"]}, elapsed={elapsed:.0f}s')

    best_full_params, _ = params_for_seed(seed, 0)
    # 실제 fold별 random_state는 fold_model_seeds에 저장한다. 대표 params에는 첫 fold seed만 들어간다.
    return {
        'seed': seed,
        'summary': summary,
        'postprocess': pp_res,
        'calibration': cal,
        'fold_models': fold_models,
        'fold_model_seeds': fold_model_seeds,
        'em_history_per_fold': em_history_per_fold,
        'best_full_params': best_full_params,
        'oof_die_pi': oof_die_pi,
        'oof_die_mu': oof_die_mu,
        'oof_die_pred_raw': oof_die_pred_raw,
        'oof_die_pred_taupi': oof_die_pred_taupi,
        'val_die_pi': val_die_pi,
        'val_die_mu': val_die_mu,
        'val_die_pred_raw': val_die_pred_raw,
        'val_die_pred_taupi': val_die_pred_taupi,
    }


## 5. seed sweep 실행

In [7]:

# RESUME=True이면 기존 summary를 읽고 이미 끝난 seed는 건너뛴다.
if RESUME and SUMMARY_PATH.exists():
    summary_df = pd.read_csv(SUMMARY_PATH)
    rows = summary_df.to_dict('records')
    done_seeds = set(summary_df['seed'].astype(int).tolist())
    best_so_far = float(summary_df['val_rmse'].min()) if len(summary_df) else float('inf')
    print(f'[재개] 기존 {len(summary_df)}개 seed 로드, 현재 best={best_so_far:.9f}')
else:
    rows = []
    done_seeds = set()
    best_so_far = float('inf')

# 핵심 실행 루프: seed 100개를 순서대로 돌린다.
# 각 seed는 5-fold ZIT 재학습이므로 시간이 오래 걸린다.
for seed in SEEDS:
    if int(seed) in done_seeds:
        print(f'[건너뜀] seed {seed}는 이미 summary에 있음')
        continue

    # seed 하나 실행 후 즉시 summary를 저장해서 중간 중단에도 결과가 남도록 한다.
    res = fit_one_seed(seed)
    rows.append(res['summary'])
    summary_df = pd.DataFrame(rows).sort_values('val_rmse').reset_index(drop=True)
    summary_df.to_csv(SUMMARY_PATH, index=False)

    # best가 갱신될 때만 무거운 fold_models.pkl과 예측 CSV를 best 폴더에 저장한다.
    seed_is_best = res['summary']['val_rmse'] < best_so_far
    if SAVE_EVERY_SEED:
        save_result_artifacts(res, OUT_DIR / 'seeds' / f'seed_{int(seed)}')
    if SAVE_BEST and seed_is_best:
        best_so_far = float(res['summary']['val_rmse'])
        save_result_artifacts(res, BEST_DIR)
        print(f'[새 best] seed={seed}, val_rmse={best_so_far:.9f}')

    del res
    gc.collect()

print('\n[완료]')
display(pd.read_csv(SUMMARY_PATH).sort_values('val_rmse').head(20))
print(f'best 산출물: {BEST_DIR}')



=== seed 2000 ===
  fold 1/5 done, model_seed=2018000, elapsed=382s
  fold 2/5 done, model_seed=2018001, elapsed=763s
  fold 3/5 done, model_seed=2018002, elapsed=1150s
  fold 4/5 done, model_seed=2018003, elapsed=1536s
  fold 5/5 done, model_seed=2018004, elapsed=1921s


[I 2026-05-27 18:21:44,630] A new study created in memory with name: no-name-5fb7f3cb-9e0e-4b24-807b-c3d29cf8ae72


[Position weights / Optuna 50t] best=0.005490, w=[0.352, 0.248, 0.232, 0.168]
[Aggregation] RMSEs: {'mean': 0.00549, 'median': 0.00549, 'max': 0.005496, 'min': 0.005496, 'trimmed_mean': 0.00549, 'weighted': 0.00549, 'Q25': 0.005491, 'Q75': 0.005491}
[Aggregation] best=median (0.005490)
[zero_clip (log)] best=0.0010 (0.005492)
[seed 2000] base_val=0.005702545, best_val=0.005701551, cal=iso_w0.75_q0.95_rq0.75_g0_p1_iqr1, iqr_outliers=1, elapsed=1926s
[저장 완료] C:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\01_zit\zit_only\seed_sweep_hp002\run_0527_174856\best
[새 best] seed=2000, val_rmse=0.005701551

=== seed 2001 ===
  fold 1/5 done, model_seed=2019009, elapsed=384s
  fold 2/5 done, model_seed=2019010, elapsed=766s
  fold 3/5 done, model_seed=2019011, elapsed=1148s
  fold 4/5 done, model_seed=2019012, elapsed=1538s
  fold 5/5 done, model_seed=2019013, elapsed=1919s
[Position weights / Optuna 50t] best=0.005491, w=[0.373, 0.235, 0.218, 0.175]
[Aggregation] RMSEs: {'mean': 0.005491, 'median': 

KeyboardInterrupt: 

## 6. 결과 확인

In [ ]:

summary = pd.read_csv(SUMMARY_PATH).sort_values('val_rmse').reset_index(drop=True)
display(summary.head(30))

# validation RMSE 최저 후보와, IQR upper outlier 1~2개 조건을 만족하는 후보를 따로 확인한다.
iqr12 = summary[summary['val_iqr_outliers'].between(1, 2)].copy()
print('[validation RMSE 기준 best]')
display(summary.head(1))

print('[IQR upper outlier 1~2개 조건 best]')
if len(iqr12):
    display(iqr12.sort_values('val_rmse').head(10))
else:
    print('아직 IQR upper outlier 1~2개 조건을 만족하는 후보가 없습니다.')

print(f'OUT_DIR  = {OUT_DIR}')
print(f'BEST_DIR = {BEST_DIR}')
